In [1]:
import pandas as pd
import numpy as np
import yaml
from linearmodels.panel import PanelOLS


with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

#Calling Dataset
file_name = "Adjusted.xlsx"
sheet_name = 'Dataset_for_Model'
file_path = f"{Setting['Output_Path_Ajusted']}/{file_name}"
df = pd.read_excel(file_path,sheet_name = sheet_name)
  
df["ln_labour"] = np.log(df["Labour_No"])
df["ln_Sale"] = np.log(df["Sale"])
df["ln_Product"] = np.log(df["Product"])
df["blackout_x_elec"] = df["Blackout"] * df['Elec_boe_intensity']


df = df.set_index(["Industry_Code", "Year"]).sort_index()
X = df[["blackout_x_elec", 'Elec_boe_intensity', "ln_labour"]]

model = PanelOLS(
    df['Log_Productivity'],
    X,
    entity_effects=True,  
    time_effects=True    
)

res = model.fit(cov_type='clustered', cluster_time=True)

print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       Log_Productivity   R-squared:                        0.2505
Estimator:                   PanelOLS   R-squared (Between):              0.5703
No. Observations:                 504   R-squared (Within):               0.4184
Date:                Thu, Apr 30 2026   R-squared (Overall):              0.5702
Time:                        15:16:50   Log-likelihood                    88.577
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      50.907
Entities:                          24   P-value                           0.0000
Avg Obs:                       21.000   Distribution:                   F(3,457)
Min Obs:                       21.000                                           
Max Obs:                       21.000   F-statistic (robust):             45.436
                            

In [2]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X0 = df.reset_index()[["blackout_x_elec", "Elec_boe_intensity", "ln_labour"]].dropna()
X0 = sm.add_constant(X0)

vif = pd.DataFrame({
    "var": X0.columns,
    "VIF": [variance_inflation_factor(X0.values, i) for i in range(X0.shape[1])]
})
print(vif)

                  var         VIF
0               const  143.871841
1     blackout_x_elec    1.694165
2  Elec_boe_intensity    1.353766
3           ln_labour    1.476040


In [3]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

u = res.resids.dropna()        
X = df[["blackout_x_elec","Elec_boe_intensity","ln_labour"]].loc[u.index]
X_bp = sm.add_constant(X)

lm, lm_pvalue, fval, f_pvalue = het_breuschpagan(u.values, X_bp.values)

print("Breusch–Pagan test")
print("LM stat:", lm)
print("LM p-value:", lm_pvalue)
print("F stat:", fval)
print("F p-value:", f_pvalue)

Breusch–Pagan test
LM stat: 31.443923925059288
LM p-value: 6.854142723874359e-07
F stat: 11.090015032795373
F p-value: 4.6536762916509045e-07


In [4]:
dfr = df.reset_index()

y_col = ["ln_Sale"]
x_cols = ["Blackout", "Elec_boe_intensity", "ln_labour", "blackout_x_elec"]

cols = y_col + x_cols

tmp = dfr[cols].replace([np.inf, -np.inf], np.nan)

desc = tmp.describe(percentiles=[0.25, 0.5, 0.75]).T
desc = desc.rename(columns={"25%": "q25", "50%": "median", "75%": "q75"})
pd.DataFrame(desc)

,count,mean,std,min,q25,median,q75,max
ln_Sale,504.0,32.066197,1.583215,27.647480,30.800582,31.981509,33.178420,36.431099
Blackout,504.0,0.070058,0.121641,0.000042,0.003729,0.017689,0.075523,0.759162
Elec_boe_intensity,504.0,0.280614,0.146989,0.003123,0.207528,0.275885,0.343318,0.999929
ln_labour,504.0,10.548917,1.145992,7.714677,9.660459,10.298532,11.594180,12.851279
blackout_x_elec,504.0,0.024976,0.049228,0.000010,0.000661,0.004720,0.021992,0.324657
